# 🍓 Çilek Hastalık + Olgunluk — YOLO26 Eğitimi (Colab Production)

Bu notebook **birleşik dataset** (10 sınıf: 7 hastalık + 3 olgunluk + sağlıklı/background) ile YOLO26 eğitir.

## ⚠️ Çalıştırmadan önce — 2 zorunlu adım

**1) GPU seçin:** Runtime → Change runtime type → Hardware accelerator: **GPU**

**2) Dataset'i Drive'a yükleyin (bir kez):** Dataset GitHub deposunda **yoktur** (418 MB).
Bilgisayarınızdaki `dataset_colab.zip` dosyasını Drive'da şu klasöre yükleyin:

```
MyDrive/StrawberryDisease/dataset_colab.zip
```

Sonra **Runtime → Run all** diyebilirsiniz. Tek elle müdahale: Drive bağlama
hücresi bir kez yetki onayı ister (Colab'ın güvenlik gereği, atlanamaz).

## 📂 Dosyalar modele nasıl veriliyor?
Görüntüler tek klasörde toplanmaz. `configs/strawberry_data.yaml` içindeki **dizin listesi**
ile 4 kaynak + augment çıktısı birlikte okunur. Label'lar, görüntü yolundaki `/images/` →
`/labels/` değişimiyle otomatik bulunur.

| Split | Görüntü | İçerik |
|---|---|---|
| train | 9.343 | 4 kaynak + augment (200'ü sağlıklı/background) |
| val | 1.341 | sadece orijinal kaynaklar (augment YOK) |
| test | 515 | sadece orijinal kaynaklar |

---

## 1️⃣ Paket kurulumu ve uyumluluk kontrolü

In [ ]:
# Colab'da SADECE ultralytics kurulur.
# NEDEN: Colab'da torch / numpy / opencv zaten kurulu ve birbiriyle uyumludur.
# Bunları elle kurmak veya yükseltmek ikili (ABI) uyumsuzluğu yaratır
# ("numpy.dtype size changed", "cv2 import error") ve runtime restart gerektirir.
# ultralytics eksik bağımlılıklarını uyumlu sürümlerle kendisi çeker.
!pip install -q "ultralytics>=8.3.200"

print('\n--- Sürüm / uyumluluk kontrolü ---')
problem = False
try:
    import numpy, torch, cv2, ultralytics
    print('numpy      :', numpy.__version__)
    print('torch      :', torch.__version__)
    print('opencv     :', cv2.__version__)
    print('ultralytics:', ultralytics.__version__)

    v = tuple(int(x) for x in ultralytics.__version__.split('.')[:3])
    if v < (8, 3, 200):
        print('\n⚠️ ultralytics sürümü YOLO26 için eski. Şunu çalıştırın: !pip install -U ultralytics')
        problem = True

    if torch.cuda.is_available():
        print('\n✅ GPU:', torch.cuda.get_device_name(0),
              f'({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)')
    else:
        print('\n⚠️ GPU YOK! Runtime > Change runtime type > GPU seçin, sonra bu hücreyi tekrar çalıştırın.')
        problem = True
except Exception as e:
    print('\n❌ Import hatası:', e)
    print('💡 Çözüm: Runtime > Restart session, sonra bu hücreyi TEKRAR çalıştırın.')
    problem = True

print('\n' + ('⚠️ Yukarıdaki uyarıyı giderin' if problem else '✅ Ortam hazır'))

## 2️⃣ Google Drive bağlantısı

Bu hücre bir kez **yetki onayı** ister (açılan pencereden hesabınızı seçin).

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/StrawberryDisease')
CHECKPOINT_DIR = DRIVE_ROOT / 'checkpoints'
RESULTS_DIR = DRIVE_ROOT / 'results'
MODELS_DIR = DRIVE_ROOT / 'best_models'
for d in (DRIVE_ROOT, CHECKPOINT_DIR, RESULTS_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print('✅ Drive hazır:', DRIVE_ROOT)

## 3️⃣ Depoyu indir / güncelle

In [ ]:
import os, subprocess
from pathlib import Path

REPO_URL = 'https://github.com/emrah1982/SmartFarmStrawberryDisease.git'
REPO_DIR = Path('/content/SmartFarmStrawberryDisease')

if REPO_DIR.exists():
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=False)
else:
    os.chdir('/content')
    subprocess.run(['git', 'clone', REPO_URL], check=True)
    os.chdir(REPO_DIR)

print('CWD:', Path.cwd())
assert (Path.cwd() / 'configs' / 'strawberry_data.yaml').exists(), \
    'configs/strawberry_data.yaml yok! Depo doğru klonlanmamış olabilir.'
print('✅ Depo hazır')

## 4️⃣ Dataset'i Drive'dan aç ve doğrula

Zip, Colab'ın **yerel diskine** açılır — Drive üzerinden eğitim çok yavaştır.
Oturum kapanınca yerel disk silinir; bu hücreyi her yeni oturumda çalıştırın (~2-3 dk).

In [ ]:
import zipfile, time
from pathlib import Path

ZIP_PATH = DRIVE_ROOT / 'dataset_colab.zip'
DATASET_DIR = REPO_DIR / 'dataset'   # data.yaml '../dataset/...' beklediği için repo kökünde olmalı

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"{ZIP_PATH} bulunamadı!\n\n"
        "Bilgisayarınızdaki dataset_colab.zip dosyasını Drive'da\n"
        "MyDrive/StrawberryDisease/ klasörüne yükleyin ve bu hücreyi tekrar çalıştırın."
    )

if DATASET_DIR.exists() and any(DATASET_DIR.iterdir()):
    print('ℹ️ dataset/ zaten mevcut, açma adımı atlandı.')
else:
    t0 = time.time()
    print(f'📦 Açılıyor: {ZIP_PATH.name} ({ZIP_PATH.stat().st_size/1e6:.0f} MB)')
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(REPO_DIR)   # zip içinde 'dataset/...' kökü var
    print(f'✅ Açıldı ({time.time()-t0:.0f} sn)')

assert DATASET_DIR.exists(), 'dataset/ oluşmadı — zip içeriğini kontrol edin.'

In [ ]:
# Eğitimden ÖNCE doğrulama: her dizin var mı, görüntü/label eşleşiyor mu?
import yaml, os
from pathlib import Path

# MUTLAK YOL ZORUNLU: Ultralytics dataset kökünü data.yaml'ın bulunduğu dizinden türetir.
# Göreli yol verilirse kökü kendi DATASETS_DIR'i altında arar → "images not found".
DATA_YAML_PATH = os.path.abspath('configs/strawberry_data.yaml')
cfg = yaml.safe_load(Path(DATA_YAML_PATH).read_text(encoding='utf-8'))
root = Path(cfg.get('path') or Path(DATA_YAML_PATH).parent)
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

names = cfg['names']
print('📁 Config:', DATA_YAML_PATH)
print('🏷️  Sınıflar:', cfg['nc'], '→', list(names.values()) if isinstance(names, dict) else names)
print()

ok = True
for split in ('train', 'val', 'test'):
    entries = cfg.get(split) or []
    entries = [entries] if isinstance(entries, str) else entries
    n_img = n_lbl = n_bg = 0
    for e in entries:
        d = (root / e).resolve()
        if not d.exists():
            print(f'❌ {split}: dizin YOK → {d}'); ok = False; continue
        ld = Path(str(d).replace('/images', '/labels'))
        imgs = [p for p in d.iterdir() if p.suffix.lower() in IMG_EXTS]
        missing = sum(1 for p in imgs if not (ld / f'{p.stem}.txt').exists())
        n_bg += sum(1 for p in imgs if (ld / f'{p.stem}.txt').exists()
                    and (ld / f'{p.stem}.txt').stat().st_size == 0)
        if missing:
            print(f"⚠️ {split}: {missing} görüntünün label'ı yok → {d.parent.parent.name}"); ok = False
        n_img += len(imgs); n_lbl += len(imgs) - missing
    print(f'{split:<6}: {len(entries)} dizin | {n_img:>5} görüntü | {n_lbl:>5} label | {n_bg} background')

print('\n' + ('✅ Dataset eğitime hazır' if ok else '❌ Sorun var — yukarıdaki uyarılara bakın'))

## 5️⃣ Eğitim konfigürasyonu

Parametreler ve gerekçeleri `configs/train_config.yaml` içindeki yorumlardadır.

> 💡 **İlk deneme için önerilir:** `OVERRIDES = {'epochs': 50, 'imgsz': 640, 'batch': 16}`
> Tam ayar (200 epoch, imgsz 1024) T4'te 10+ saat sürer ve ücretsiz Colab oturumu kopabilir.
> Kısa turla sınıf bazlı metrikleri birkaç saatte görüp sonra tam eğitime geçin.

In [ ]:
import yaml
from pathlib import Path

TRAIN_CONFIG = yaml.safe_load(Path('configs/train_config.yaml').read_text(encoding='utf-8'))

# Hızlı ilk tur için: {'epochs': 50, 'imgsz': 640, 'batch': 16}
# Bellek yetmezse:    {'batch': 4}
OVERRIDES = {}
TRAIN_CONFIG.update(OVERRIDES)

# Sonuçlar doğrudan Drive'a yazılsın (oturum kopsa bile kaybolmaz)
TRAIN_CONFIG['project'] = str(RESULTS_DIR)

for k in ('model', 'epochs', 'batch', 'imgsz', 'optimizer', 'cos_lr', 'patience', 'save_period'):
    print(f'  {k}: {TRAIN_CONFIG.get(k)}')
print('\n  sonuç dizini:', TRAIN_CONFIG['project'])

## 6️⃣ Eğitim

Her `save_period` epoch'ta checkpoint Drive'a yazılır. Oturum koparsa
aşağıdaki "devam et" hücresiyle kaldığı yerden sürdürebilirsiniz.

In [ ]:
from ultralytics import YOLO
import time, shutil
from pathlib import Path

model = YOLO(TRAIN_CONFIG['model'])   # yolo26s.pt ilk çalıştırmada otomatik indirilir

t0 = time.time()
results = model.train(data=DATA_YAML_PATH, **TRAIN_CONFIG)
print(f'\n✅ Eğitim bitti: {(time.time()-t0)/3600:.2f} saat')

results_dir = Path(TRAIN_CONFIG['project']) / TRAIN_CONFIG['name']
best_path = results_dir / 'weights' / 'best.pt'
print('📊 Sonuçlar:', results_dir)

if best_path.exists():
    dest = MODELS_DIR / f"best_{TRAIN_CONFIG['name']}.pt"
    shutil.copy(best_path, dest)
    print("🏆 En iyi model Drive'a kopyalandı:", dest)

In [ ]:
# (SADECE oturum koptuysa çalıştırın) Kaldığı yerden devam et
from ultralytics import YOLO
from pathlib import Path

last_ckpt = Path(TRAIN_CONFIG['project']) / TRAIN_CONFIG['name'] / 'weights' / 'last.pt'
if last_ckpt.exists():
    print('🔄 Devam ediliyor:', last_ckpt)
    YOLO(str(last_ckpt)).train(resume=True)
else:
    print('ℹ️ last.pt yok — devam edilecek eğitim bulunmuyor.')

## 7️⃣ Değerlendirme — sınıf bazlı (ticari karar buradan verilir)

Genel mAP tek başına yanıltıcıdır: ortalama iyi görünürken tek bir hastalıkta recall
çok düşük olabilir. Az örnekli sınıflara (Anthracnose Fruit Rot, Powdery Mildew Fruit)
ayrıca bakın.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

best_path = Path(TRAIN_CONFIG['project']) / TRAIN_CONFIG['name'] / 'weights' / 'best.pt'
if not best_path.exists():
    print('⚠️ best.pt bulunamadı — önce eğitim hücresini çalıştırın.')
else:
    model = YOLO(str(best_path))
    m = model.val(data=DATA_YAML_PATH)   # eğitimdekiyle AYNI config

    print('\n' + '='*58)
    print(f'GENEL  mAP50: {m.box.map50:.4f} | mAP50-95: {m.box.map:.4f} | '
          f'P: {m.box.mp:.4f} | R: {m.box.mr:.4f}')
    print('='*58)
    print(f"\n{'Sınıf':<24}{'P':>8}{'R':>8}{'mAP50':>9}")
    cls_names = model.names
    for i, c in enumerate(m.box.ap_class_index):
        flag = '  ⚠️ düşük recall' if m.box.r[i] < 0.75 else ''
        print(f'{cls_names[int(c)]:<24}{m.box.p[i]:>8.3f}{m.box.r[i]:>8.3f}{m.box.ap50[i]:>9.3f}{flag}')
    print('\n💡 confusion_matrix.png: hangi hastalık hangisiyle karışıyor?')

In [ ]:
# Eğitim grafikleri ve confusion matrix
from IPython.display import Image, display
from pathlib import Path

rd = Path(TRAIN_CONFIG['project']) / TRAIN_CONFIG['name']
for f in ('results.png', 'confusion_matrix.png', 'val_batch0_pred.jpg'):
    p = rd / f
    if p.exists():
        print(f)
        display(Image(filename=str(p)))

## 8️⃣ Örnek tahminler

Yüksek çözünürlüklü saha fotoğraflarında küçük lezyonlar için
`scripts/sahi_predict.py` (dilimli inference) kullanın.

In [ ]:
import cv2, yaml
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

cfg = yaml.safe_load(Path(DATA_YAML_PATH).read_text(encoding='utf-8'))
root = Path(cfg.get('path') or Path(DATA_YAML_PATH).parent)
val_dirs = cfg['val'] if isinstance(cfg['val'], list) else [cfg['val']]

imgs = []
for d in val_dirs:
    p = (root / d).resolve()
    if p.exists():
        imgs += sorted(p.glob('*.jpg'))[:2]

best_path = Path(TRAIN_CONFIG['project']) / TRAIN_CONFIG['name'] / 'weights' / 'best.pt'
if imgs and best_path.exists():
    model = YOLO(str(best_path))
    for ip in imgs[:5]:
        r = model(str(ip), verbose=False)[0]
        plt.figure(figsize=(11, 7))
        plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off')
        plt.title(ip.name); plt.show()
        counts = {}
        for b in r.boxes:
            n = model.names[int(b.cls[0])]
            counts[n] = counts.get(n, 0) + 1
        print(f"{ip.name}: {len(r.boxes)} tespit → {counts or 'yok'}\n" + '-'*50)
else:
    print('⚠️ Görüntü veya model bulunamadı.')

---
## 📝 Notlar

**Sonuçlar nerede?** Eğitim doğrudan Drive'a yazar:
```
MyDrive/StrawberryDisease/
├── results/strawberry_exp/        # grafikler, confusion matrix, weights/
└── best_models/best_strawberry_exp.pt
```

**Sık karşılaşılan sorunlar**

| Sorun | Çözüm |
|---|---|
| `images not found` | `data=` mutlak yol mu? (bu notebook otomatik yapar) |
| CUDA out of memory | `OVERRIDES = {'batch': 4}` veya `{'imgsz': 640, 'batch': 16}` |
| `numpy.dtype size changed` / cv2 import hatası | Runtime → Restart session, sonra 1️⃣ hücresini tekrar çalıştırın |
| `yolo26s.pt` yüklenemiyor | `!pip install -U ultralytics` (>=8.3.200 gerekir) |
| Oturum koptu | 6️⃣ bölümündeki "devam et" hücresi |
| Eğitim çok yavaş | GPU seçili mi? 1️⃣ hücresini kontrol edin |

**Dataset güncellenirse:** Bilgisayarda yeni `dataset_colab.zip` oluşturup Drive'daki
dosyanın üzerine yazın, Colab'da `dataset/` klasörünü silip 4️⃣ hücresini tekrar çalıştırın.